In [1]:
import kagglehub, os, glob, hashlib
IMG  = kagglehub.dataset_download("ashwanisingh997/ham10000-images-1229692")
SPL  = kagglehub.dataset_download("ashwanisingh997/ham10000-splits-1229692")
CODE = kagglehub.dataset_download("ashwanisingh997/thesis-code-1229692")
CKPT = kagglehub.dataset_download("ashwanisingh997/run1-checkpoints-1229692")
SYN  = kagglehub.dataset_download("ashwanisingh997/run1-synthetic-headline-1229692")

src = {f: open(f"{CODE}/{f}").read() for f in ["memorization.py", "fid.py", "checkpointing.py", "diffusion.py"]}
assert "def verify_equivalence" in src["memorization.py"] and "class InceptionFeatures" in src["fid.py"], "code is not v5"
assert 'format={"jpg"' in src["checkpointing.py"] and "pipe.vae.enable_slicing()" in src["diffusion.py"], "sampling fixes missing"
print("code: v5 (6c8939b) confirmed")

rows = []
for dp, _, files in os.walk(CKPT):
    for f in files:
        full = os.path.join(dp, f); h = hashlib.sha256()
        with open(full, "rb") as fh:
            for ch in iter(lambda: fh.read(1 << 20), b""): h.update(ch)
        rows.append((os.path.relpath(full, CKPT).replace(os.sep, "/"), h.hexdigest()))
rows.sort()
dig = hashlib.sha256("\n".join(f"{p}|{s}" for p, s in rows).encode()).hexdigest()
assert dig == "81280427151b71c7eb1cc79d78415265d6924ec30c27210ee0bd0c708672dbf2", "CHECKPOINTS DIFFER"
print("checkpoints: MATCH")

SYN_ROOT = f"{SYN}/synthetic/run1_cloud"
ARMS = ["gan_final", "lora_final"]
for arm in ARMS:
    for c in ["akiec", "bcc", "df", "mel", "vasc"]:
        assert len(glob.glob(f"{SYN_ROOT}/{arm}/{c}/*.png")) == 1000, (arm, c)
print("synthetic headline set: 10,000 confirmed")

code: v5 (6c8939b) confirmed
checkpoints: MATCH
synthetic headline set: 10,000 confirmed


In [2]:
!pip uninstall -y -q torchao
!pip install -q --no-deps lpips pytorch-fid
import importlib; importlib.invalidate_caches()
import peft.import_utils as piu
if hasattr(piu.is_torchao_available, "cache_clear"): piu.is_torchao_available.cache_clear()
assert piu.is_torchao_available() is False
import lpips, pytorch_fid
print("CELL 2 OK")

CELL 2 OK


In [3]:
import sys, shutil, torch, yaml, numpy as np, pandas as pd
WORK = "/kaggle/working/repo"
os.makedirs(f"{WORK}/src", exist_ok=True); os.makedirs(f"{WORK}/config", exist_ok=True)
for f in os.listdir(CODE):
    shutil.copy(f"{CODE}/{f}", f"{WORK}/config/{f}" if f.endswith(".yaml") else f"{WORK}/src/{f}")
sys.path.insert(0, WORK); os.chdir(WORK)
print(torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
assert torch.__version__.startswith("2.11")

cfg = yaml.safe_load(open("config/config.yaml"))
from src.run_registry import init_run
root = init_run("run1_cloud", root="/kaggle/working/results", git_hash="6c8939b")
os.makedirs(f"{root}/metrics", exist_ok=True); os.makedirs(f"{root}/samples", exist_ok=True)

from src.data import build_image_index, generator_source_frame, AUG_CLASSES
frame = generator_source_frame(f"{SPL}/train.csv")
index = build_image_index(IMG)
ref_paths = [index[i] for i in frame.image_id]
assert len(ref_paths) == 1526
print("reference set: 1,526 generator source images")

2.11.0+cu128 | GPU: Tesla T4
[registry] run1_cloud (PRIMARY - Kaggle free-tier GPU, matches proposal Table 8.2)
[registry] segment 1 on Tesla T4 | torch 2.11.0+cu128 | cuda 12.8
[registry] writing to /kaggle/working/results/run1_cloud
[data] generator source frame: 1526 images across ['akiec', 'bcc', 'df', 'mel', 'vasc']
dx
mel      773
bcc      361
akiec    222
vasc      99
df        71
[data] indexed 10015 images under /kaggle/input/datasets/ashwanisingh997/ham10000-images-1229692
reference set: 1,526 generator source images


In [4]:
import gc
from PIL import Image
from src.gan import Generator
from src.diffusion import BASE, build_pipeline, generate_diffusion, generate_gan_images

SMOKE = False   # True: quick test. False: the real run.
DF_TARGET = 1002 if SMOKE else 2000
NSYN = 20 if SMOKE else 1000
TAG = "_smoke" if SMOKE else ""
DFW = f"/kaggle/working/synthetic_df{TAG}"
CK, LOG = f"{CKPT}/checkpoints", f"{root}/logs"

for arm in ARMS:   # reuse stored DF images 0-999 (D25)
    if not os.path.exists(f"{DFW}/{arm}/df"):
        shutil.copytree(f"{SYN_ROOT}/{arm}/df", f"{DFW}/{arm}/df")

def repro(arm, k=5):   # regenerated vs stored, mean |diff| per image on 0-255
    return [round(float(np.abs(np.asarray(Image.open(f"/kaggle/working/_repro/{arm}/df/df_{i:04d}.png"), float)
             - np.asarray(Image.open(f"{SYN_ROOT}/{arm}/df/df_{i:04d}.png"), float)).mean()), 3) for i in range(k)]

G = Generator(latent_dim=cfg["gan"]["latent_dim"])
G.load_state_dict(torch.load(f"{CK}/gan_final.pt", map_location="cpu", weights_only=False)["models"]["G"])
generate_gan_images(G, ["df"], 5, "/kaggle/working/_repro/gan_final", latent_dim=cfg["gan"]["latent_dim"], base_seed=42)
r = repro("gan_final"); print("GAN reproducibility:", r); assert max(r) < 2.0
generate_gan_images(G, ["df"], DF_TARGET, f"{DFW}/gan_final", latent_dim=cfg["gan"]["latent_dim"],
                    base_seed=42, manifest_path=f"{LOG}/sampling_df_extra_gan{TAG}.json")
del G; gc.collect(); torch.cuda.empty_cache()

from diffusers import UNet2DConditionModel
from peft import PeftModel
unet = UNet2DConditionModel.from_pretrained(BASE, subfolder="unet", torch_dtype=torch.float16)
unet = PeftModel.from_pretrained(unet, f"{CK}/lora_final").merge_and_unload().to(torch.float16)
pipe = build_pipeline(unet)
s = cfg["diffusion"]["sampler"]
kw = dict(base_seed=42, steps=s["num_inference_steps"], guidance=s["guidance_scale"],
          size=cfg["data"]["image_size_diffusion"], target_size=cfg["data"]["image_size_classifier"])
generate_diffusion(pipe, ["df"], 5, "/kaggle/working/_repro/lora_final", **kw)
r = repro("lora_final"); print("Diffusion reproducibility:", r); assert max(r) < 2.0
generate_diffusion(pipe, ["df"], DF_TARGET, f"{DFW}/lora_final",
                   manifest_path=f"{LOG}/sampling_df_extra_lora{TAG}.json", session_limit_hours=10.0, **kw)
del pipe, unet; gc.collect(); torch.cuda.empty_cache()

for arm in ARMS:
    assert len(glob.glob(f"{DFW}/{arm}/df/*.png")) == DF_TARGET, arm
print(f"DF sets: {DF_TARGET} per arm, nested on stored images 0-999")

[sample] {
  "out_dir": "/kaggle/working/_repro/gan_final",
  "classes": [
    "df"
  ],
  "n_per_class_target": 5,
  "counts_on_disk": {
    "df": 5
  },
  "base_seed": 42,
  "images_generated_this_session": 5,
  "images_skipped_existing": 0,
  "session_seconds": 1.75,
  "seconds_per_image": 0.35,
  "complete": true,
  "device": "Tesla T4"
}
GAN reproducibility: [0.0, 0.0, 0.0, 0.0, 0.0]
[sample] df: 50 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 100 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 150 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 200 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 250 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 300 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 350 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 400 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 450 made, 1000 skipped, 0.01s/image, ~0.00h left
[sample] df: 500 made, 1000 skipped, 0.01s/image, ~

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

unet/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

unet/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

There are modules in UNet2DConditionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


... (3 outputs left out)

Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


[sample] {
  "out_dir": "/kaggle/working/_repro/lora_final",
  "classes": [
    "df"
  ],
  "n_per_class_target": 5,
  "counts_on_disk": {
    "df": 5
  },
  "base_seed": 42,
  "images_generated_this_session": 5,
  "images_skipped_existing": 0,
  "session_seconds": 23.67,
  "seconds_per_image": 4.734,
  "complete": true,
  "device": "Tesla T4"
}
Diffusion reproducibility: [0.0, 0.0, 0.0, 0.0, 0.0]
[sample] df: 50 made, 1000 skipped, 4.94s/image, ~1.30h left
[sample] df: 100 made, 1000 skipped, 4.97s/image, ~1.24h left
[sample] df: 150 made, 1000 skipped, 4.97s/image, ~1.17h left
[sample] df: 200 made, 1000 skipped, 4.98s/image, ~1.11h left
[sample] df: 250 made, 1000 skipped, 4.98s/image, ~1.04h left
[sample] df: 300 made, 1000 skipped, 4.98s/image, ~0.97h left
[sample] df: 350 made, 1000 skipped, 4.98s/image, ~0.90h left
[sample] df: 400 made, 1000 skipped, 4.98s/image, ~0.83h left
[sample] df: 450 made, 1000 skipped, 4.98s/image, ~0.76h left
[sample] df: 500 made, 1000 skipped, 4.98s

In [5]:
from src.fid import InceptionFeatures, fid
inc = InceptionFeatures()
real = {c: inc([p for p, d in zip(ref_paths, frame.dx) if d == c]) for c in AUG_CLASSES}
rows = []
for arm in ARMS:
    syn = {c: inc(sorted(glob.glob(f"{SYN_ROOT}/{arm}/{c}/*.png"))[:NSYN]) for c in AUG_CLASSES}
    for c in AUG_CLASSES:
        rows.append({"arm": arm, "class": c, "n_real": len(real[c]), "n_syn": len(syn[c]), "fid": fid(real[c], syn[c])})
    rows.append({"arm": arm, "class": "ALL", "n_real": 1526, "n_syn": sum(len(v) for v in syn.values()),
                 "fid": fid(np.concatenate(list(real.values())), np.concatenate(list(syn.values())))})
fid_df = pd.DataFrame(rows); fid_df.to_csv(f"{root}/metrics/fid{TAG}.csv", index=False)
print(fid_df.pivot(index="class", columns="arm", values="fid").round(2).to_string())
del inc; gc.collect(); torch.cuda.empty_cache()

Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to /root/.cache/torch/hub/checkpoints/pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:00<00:00, 191MB/s]


arm    gan_final  lora_final
class                       
ALL       206.71      137.51
akiec     256.03      169.66
bcc       227.69      149.01
df        280.59      204.89
mel       216.75      170.79
vasc      237.24      237.66


In [6]:
from src.memorization import (LPIPSEmbedder, load_images, embed_paths, real_to_real,
                              synthetic_to_real, calibrate_threshold, summarise, save_worst_pairs)
emb = LPIPSEmbedder()
x = load_images(ref_paths[:32]); emb.verify_equivalence(x, x.flip(0))
ref_emb = embed_paths(emb, ref_paths)

r2r, j = real_to_real(ref_emb, frame.lesion_id.values)
assert (frame.lesion_id.values[j] != frame.lesion_id.values).all(), "same-lesion neighbour found"
tau = calibrate_threshold(r2r, cfg["audit"]["percentile"])
pd.DataFrame({"image_id": frame.image_id, "dx": frame.dx, "nn_dist": r2r,
              "nn_image_id": frame.image_id.values[j]}).to_csv(f"{root}/metrics/audit_real_to_real{TAG}.csv", index=False)

def audit_set(arm, paths, cls):
    d, k = synthetic_to_real(emb, paths, ref_emb)
    return pd.DataFrame({"arm": arm, "dx": cls, "path": paths, "nn_dist": d, "nn_index": k,
                         "nn_image_id": frame.image_id.values[k], "nn_lesion": frame.lesion_id.values[k],
                         "nn_dx": frame.dx.values[k]})

head = pd.concat([audit_set(a, sorted(glob.glob(f"{SYN_ROOT}/{a}/{c}/*.png"))[:NSYN], c)
                  for a in ARMS for c in AUG_CLASSES], ignore_index=True)
head.to_csv(f"{root}/metrics/audit_headline_per_image{TAG}.csv", index=False)
summ = summarise(head, tau); summ.to_csv(f"{root}/metrics/audit_headline_summary{TAG}.csv", index=False)
print(f"\nthreshold tau = {tau:.4f}\n"); print(summ.round(4).to_string(index=False))
save_worst_pairs(head, ref_paths, f"{root}/samples/worst_pairs_headline{TAG}.png", n=cfg["audit"]["n_worst_pairs_to_plot"])

n_df_real = int((frame.dx == "df").sum())
abl = []
for arm in ARMS:
    a = audit_set(arm, sorted(glob.glob(f"{DFW}/{arm}/df/*.png")), "df")
    a.to_csv(f"{root}/metrics/audit_df_ablation_per_image_{arm}{TAG}.csv", index=False)
    for n in ([10, 1002] if SMOKE else [100, 250, 500, 1000, 2000]):
        d = a.nn_dist.values[:n]; k = int((d < tau).sum())
        abl.append({"arm": arm, "n_synthetic": n, "near_duplicates": k, "rate": k / n,
                    "share_of_df_training": k / (n_df_real + n), "min_dist": d.min(), "median": float(np.median(d))})
abl = pd.DataFrame(abl); abl.to_csv(f"{root}/metrics/audit_df_ablation{TAG}.csv", index=False)
print("\nDF ablation (D24)"); print(abl.round(4).to_string(index=False))

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 182MB/s]


[audit] LPIPS equivalence: max |direct - embedded| = 1.79e-07 over 32 pairs (tolerance 0.001)
[audit] threshold tau = 0.1333 (1.0th percentile of 1526 real-to-real distances)

threshold tau = 0.1333

       arm class    n  near_duplicates  rate    min    p01    p05  median
 gan_final akiec 1000                0   0.0 0.1963 0.2065 0.2282  0.2876
 gan_final   bcc 1000                0   0.0 0.1719 0.2037 0.2269  0.2899
 gan_final    df 1000                0   0.0 0.1439 0.1560 0.1720  0.2702
 gan_final   mel 1000                0   0.0 0.1791 0.2061 0.2389  0.3144
 gan_final  vasc 1000                0   0.0 0.1826 0.1975 0.2180  0.2749
lora_final akiec 1000                0   0.0 0.1522 0.1916 0.2130  0.2711
lora_final   bcc 1000                0   0.0 0.1790 0.1964 0.2164  0.2676
lora_final    df 1000                0   0.0 0.1468 0.1868 0.2003  0.2476
lora_final   mel 1000                0   0.0 0.1669 0.1891 0.2124  0.2670
lora_final  vasc 1000                0   0.0 0.1432 0.1771 0

In [7]:
shutil.make_archive("/kaggle/working/run1_audit_results", "zip", "/kaggle/working/results")
print("saved: /kaggle/working/run1_audit_results.zip")

saved: /kaggle/working/run1_audit_results.zip
